# 007-search-prior-and-gate

# Imports

In [ ]:
import json
import os
import time
from dataclasses import asdict

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lopo import TuningGrid, tuned_config
from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import benchmark_draws, injected_priors, score_draws, summarise
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "007-search-prior-and-gate"
NOTE = ""
LOG_RUN = True

WINDOW = "rect"                   # "rect" | "round"
SEARCH_SCALE = 1.7                # rect window: patch is scale x footprint
SEARCH_RADIUS_M = None            # round window: drone centre may move this far
PRIOR_SIGMA_M = None              # None: flat prior
GATE = "none"                     # "none" | "full" | "accept-reject"
TEMPERATURE_GRID = [0.25, 0.5, 1.0, 2.0]
S_MAX_GRID_M = [2.0, 3.0, 5.0]
C_MIN_GRID = [0.01, 0.03, 0.05]
TUNING_NOISE_M = 10.0
MIN_APPLIED_SHARE = 0.5
TEMPERATURE_BY = "mean-error"     # "mean-error" (pre-registered) | "gate": chosen jointly with the gate

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"
MATCHING = "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
TIMING_DEVICE = "cpu"
TIMING_FRAMES = 8
TIMING_REPEATS = 3
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"

# Data

In [ ]:
set_seed(SEED)

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "window": WINDOW, "search_scale": SEARCH_SCALE, "search_radius_m": SEARCH_RADIUS_M,
    "prior_sigma_m": PRIOR_SIGMA_M, "gate": GATE, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
    "tuning": {"temperatures": TEMPERATURE_GRID, "s_max_m": S_MAX_GRID_M, "c_min": C_MIN_GRID,
               "noise_m": TUNING_NOISE_M, "min_applied_share": MIN_APPLIED_SHARE, "scheme": "leave-one-place-out",
               "temperature_by": TEMPERATURE_BY},
}

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
{place: len(members) for place, members in places.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)

In [ ]:
features = {place: SegmentationFeatures(model, model, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)
            for place in PLACES}
base_config = correlator_config({**RUN_CONFIG, "gate": "none"})   # the gate and its values come from tuning
correlators = {place: PositionCorrelator(features[place], base_config) for place in PLACES}
base_config

# Run

In [ ]:
draws = {place: [d for noise_m in NOISE_M
                 for d in benchmark_draws(correlators[place], places[place], noise_m, N_DIRECTIONS, SEED)]
         for place in PLACES}
{place: len(d) for place, d in draws.items()}

In [ ]:
def tuning_place(held_out: str) -> str:
    """Leave-one-place-out: a held-out place is scored with values chosen on the other one."""
    return next(place for place in PLACES if place != held_out)

In [ ]:
grid = TuningGrid(tuple(TEMPERATURE_GRID), tuple(S_MAX_GRID_M), tuple(C_MIN_GRID), TUNING_NOISE_M, MIN_APPLIED_SHARE,
                  TEMPERATURE_BY)
folds = {place: tuned_config(draws[tuning_place(place)], base_config, GATE, grid) for place in PLACES}
fold_configs = {place: config for place, (config, _) in folds.items()}
{place: (c.temperature, c.gate_s_max_m, c.gate_c_min) for place, c in fold_configs.items()}

In [ ]:
tuning = pd.concat([t.assign(held_out=place) for place, (_, t) in folds.items()], ignore_index=True)
tuning.round(3)

In [ ]:
records = pd.concat([score_draws(draws[place], fold_configs[place]).assign(place=place) for place in PLACES],
                    ignore_index=True)
records["kept_prior"] = records["decision"] == "reject"
records["harm"] = records["error_m"] > records["prior_error_m"] + 2.0
records.groupby(["place", "noise_m"])[["error_m", "harm", "kept_prior", "reachable"]].mean().round(3)

In [ ]:
pd.crosstab([records["noise_m"], records["place"]], records["decision"], normalize="index").round(3)

In [ ]:
def run_metrics(records: pd.DataFrame) -> dict[str, float]:
    """005/006's metrics (harm and decision shares included), plus the share of draws that kept the prior."""
    subsets = {"all": {f.name for f in frames}, **{p.split("-")[0]: {f.name for f in m} for p, m in places.items()}}
    metrics = summarise(records, subsets, PRIMARY_NOISE_M)
    for noise_m, group in records.groupby("noise_m"):
        metrics[f"kept_prior_share_at_{noise_m:g}m"] = float(group["kept_prior"].mean())
    metrics["kept_prior_share"] = metrics[f"kept_prior_share_at_{PRIMARY_NOISE_M:g}m"]
    return metrics

In [ ]:
metrics = run_metrics(records)
{k: round(v, 3) for k, v in metrics.items() if "_at_" not in k}

# Timing

In [ ]:
def median_ms_per_correction(correlator: PositionCorrelator, members: list[DroneFrame]) -> float:
    """Median over repeats of the mean wall time of one correction, drone inference included (cache cleared)."""
    priors = [injected_priors(f.position, TUNING_NOISE_M, N_DIRECTIONS, 0.0)[0][1] for f in members]
    repeats = []
    for _ in range(TIMING_REPEATS):
        correlator._drone_maps.clear()
        start = time.perf_counter()
        for frame, prior in zip(members, priors):
            correlator.correct(frame, prior)
        repeats.append((time.perf_counter() - start) * 1000 / len(members))
    return float(np.median(repeats))

In [ ]:
timing_place = PLACES[0]
timing_model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, TIMING_DEVICE, SEG_ARCHITECTURE)
timing_features = SegmentationFeatures(timing_model, timing_model, maps[timing_place], SEG_CHANNELS,
                                       MATCH_GSD_M, MATCH_GSD_M, TIMING_DEVICE)
timing_correlator = PositionCorrelator(timing_features, fold_configs[timing_place])
metrics["ms_per_correction"] = median_ms_per_correction(timing_correlator, places[timing_place][:TIMING_FRAMES])
metrics["ms_per_correction"]

# Plots

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(6, 4))
for noise_m, group in records.groupby("noise_m"):
    errors = np.sort(group["error_m"].to_numpy())
    line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
    ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
ax.set_title(f"{WINDOW} window, sigma {PRIOR_SIGMA_M}, gate {GATE}"); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

In [ ]:
per_frame = records[records["noise_m"] == PRIMARY_NOISE_M].groupby("frame")["error_m"].mean().sort_values()
fig_frames, ax = plt.subplots(figsize=(12, 3.5))
per_frame.plot.bar(ax=ax, width=.8)
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title("per frame, 8 directions each")
fig_frames.tight_layout()

# Log

In [ ]:
def per_fold(field: str) -> dict[str, float | None]:
    """A tuned value per held-out place; None where the run does not use it."""
    used = {"temperature": PRIOR_SIGMA_M is not None or GATE != "none",
            "gate_s_max_m": GATE != "none", "gate_c_min": GATE != "none"}[field]
    return {place: getattr(config, field) if used else None for place, config in fold_configs.items()}

In [ ]:
RUN_CONFIG.update({field: per_fold(field) for field in ("temperature", "gate_s_max_m", "gate_c_min")})
RUN_CONFIG["timing_device"] = TIMING_DEVICE

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_cdf, fig_frames],
              seed=SEED, device=DEVICE) if LOG_RUN else {"run": "dry"}
records.to_csv(artifacts / f"records-{run['run']}.csv", index=False)
tuning.to_csv(artifacts / f"tuning-{run['run']}.csv", index=False)
run["run"]